# Developing Axolotl Telencephalon — fixed 99-epoch SpaDOT example

This notebook follows the preprocessing, training, and spatial analysis steps
of the original Chicken Heart tutorial. The raw data are in
`AxolotlTelencephalon_data/` (see the file path below); the frozen feature panel is in
`fixed99_panels/AxolotlTelencephalon/`.

The **Stereo-seq** dataset contains t0, t1, and t2. The evaluated
pseudoregion panel selected **500 / 500 / 1,000 genes per timepoint** and uses their
sorted union of **1,250 genes** for every timepoint. The lists were
selected without annotation labels or marker identities: mt/ribosomal filtering,
R SPARK-X FDR candidates, 12 expression/spatial KMeans pseudoregions per
timepoint, balanced one-versus-rest contrasts, and protection of the top 100
SPARK-X genes per timepoint. These panel sizes were chosen after development
evaluation and should be validated on new samples. The complete selection
procedure is implemented in `select_pseudoregion_svgs.py` and
`run_filtered_sparkx.R`; see `fixed99_panels/README.md` for the replay steps.

Use the checked-out SpaDOT review source with Python 3.9 and the dependencies
in `pyproject.toml`. For closest reproducibility, start Jupyter with
`PYTHONHASHSEED=2026`, `CUBLAS_WORKSPACE_CONFIG=:4096:8`, and numerical thread
limits of 1. The notebook writes to `fixed99_runs/AxolotlTelencephalon/`, leaving the
historical example outputs untouched. The panel preparation uses the frozen
gene identities, spot order, and coordinates; generic `SpaDOT.preprocess`
feature selection does **not** reproduce this panel. Raw `.h5ad` files are
not bundled; place the file at the path below, or set
`SPADOT_EXAMPLES_DATA_DIR` to a directory containing the data subfolder.


In [ ]:
import os
import sys
from pathlib import Path
from argparse import Namespace

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

examples = Path.cwd()
if not (examples / "fixed99_panels").is_dir():
    examples = examples / "examples"
if not (examples / "fixed99_panels").is_dir():
    raise FileNotFoundError("Run this notebook from the repository root or examples/")
sys.path.insert(0, str(examples.parent))
sys.path.insert(0, str(examples))

import torch
from threadpoolctl import threadpool_limits
import SpaDOT
from fixed99_example import prepare_panel

torch.use_deterministic_algorithms(True, warn_only=False)
torch.set_num_threads(1)

panel_dir = examples / "fixed99_panels" / "AxolotlTelencephalon"
data_root = Path(os.environ.get("SPADOT_EXAMPLES_DATA_DIR", str(examples)))
raw_file = data_root / "AxolotlTelencephalon_data" / "Axolotl.h5ad"
output_dir = examples / "fixed99_runs" / "AxolotlTelencephalon"
output_dir.mkdir(parents=True, exist_ok=True)
device = "cuda:0"  # Choose an available GPU, or use "cpu" for a slower run.
print("SpaDOT source:", SpaDOT.__file__)


## 1. Preprocess and select features

The model input contains the sorted union of the selected genes. Each timepoint is normalized to a total of `1e-4`, transformed by `log1p`, and scaled separately. Raw selected counts remain in `layers["counts"]`.


In [ ]:
# Step 1: select the frozen SVG union from raw counts and preprocess each timepoint.
input_file = output_dir / "preprocessed_fixed99.h5ad"
with threadpool_limits(limits=1):
    prepare_panel(raw_file, panel_dir, input_file)
print("Prepared:", input_file)


## 2. Train SpaDOT

The default review configuration executes epochs 1 through 99, with `kl_cycle_epochs: 30`, seed 2026, and Python/NumPy OT.


In [ ]:
# Step 2: train for epochs 1–99 with a 30-epoch KL cycle.
# The saved latent and model are from the executed final epoch.
train_args = Namespace(data=str(input_file), output_dir=str(output_dir),
                       prefix="", config=None, save_model=True, device=device)
with threadpool_limits(limits=1):
    SpaDOT.train(train_args)


## 3. Analyze spatial domains

This example uses fixed region counts [6, 7, 8] and final KMeans with seed 2026 and `n_init=10`. The analysis writes spatial-domain images and transition plots into the same output directory.


In [ ]:
# Step 3: infer spatial domains and temporal transition plots.
analyze_args = Namespace(data=str(output_dir / "latent.h5ad"),
                         output_dir=str(output_dir), prefix="",
                         n_clusters=[6, 7, 8], seed=2026, n_init=10)
with threadpool_limits(limits=1):
    SpaDOT.analyze(analyze_args)


In [ ]:
from IPython.display import Image, display

for tp in ['t0', 't1', 't2']:
    display(Image(filename=str(output_dir / f"{tp}_domains.png")))
